# 03 · Launch-campaign purchase propensity (binary logistic regression)
statsmodels for inference, scikit-learn pipeline for out-of-sample evaluation.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from neatnest import propensity
cp = pd.read_csv('../data/campaign.csv')
print(cp.shape, 'conversion', round(cp.purchased.mean(), 4))

(5000, 10) conversion 0.0786


In [2]:
res, table = propensity.statsmodels_logit(cp)
print(res.summary2().tables[0])
table.round(4)

                     0                 1                  2           3
0               Model:             Logit            Method:         MLE
1  Dependent Variable:         purchased  Pseudo R-squared:       0.120
2                Date:  2026-10-03 09:41               AIC:   2442.2104
3    No. Observations:              5000               BIC:   2507.3823
4            Df Model:                 9    Log-Likelihood:     -1211.1
5        Df Residuals:              4990           LL-Null:     -1376.7
6           Converged:            1.0000       LLR p-value:  6.3146e-66
7      No. Iterations:            7.0000             Scale:      1.0000


,coef,std_err,p_value,odds_ratio,ci_low,ci_high,significant
age,0.0103,0.0070,0.1404,1.0104,0.9966,1.0244,False
monthly_income_k,0.0006,0.0012,0.6045,1.0006,0.9982,1.0031,False
digital_engagement_score,0.0289,0.0033,0.0000,1.0294,1.0227,1.0361,True
prev_home_org_purchases,0.4046,0.0399,0.0000,1.4987,1.3860,1.6206,True
site_visits,0.0913,0.0276,0.0009,1.0956,1.0379,1.1565,True
fit_quiz_completed,0.8960,0.1171,0.0000,2.4497,1.9475,3.0816,True
source_Google search,0.1577,0.1683,0.3487,1.1709,0.8418,1.6285,False
source_Housing-society activation,0.2675,0.2014,0.1841,1.3067,0.8806,1.9389,False
source_Meta paid social,0.1335,0.1544,0.3873,1.1428,0.8444,1.5466,False


In [3]:
ev = propensity.evaluate(cp)
print('CV AUC', round(ev['cv_auc_mean'], 3), '+/-', round(ev['cv_auc_std'], 3))
ev['holdout']

CV AUC 0.743 +/- 0.018


{'auc': 0.7535696570294784,
 'accuracy': 0.6936,
 'precision': 0.15990453460620524,
 'recall': 0.6836734693877551,
 'f1': 0.25918762088974856,
 'n_test': 1250}

In [4]:
ev['lift'].round(3)

,prospects,buyers,conv_rate,lift,cum_capture
decile,,,,,
1,125,34,0.272,3.469,0.347
2,125,18,0.144,1.837,0.531
3,125,13,0.104,1.327,0.663
4,125,9,0.072,0.918,0.755
5,125,8,0.064,0.816,0.837
6,125,5,0.040,0.510,0.888
7,125,1,0.008,0.102,0.898
8,125,2,0.016,0.204,0.918
9,125,5,0.040,0.510,0.969


In [5]:
propensity.score_prospect(ev['model'], age=32, monthly_income_k=90, digital_engagement_score=65,
    prev_home_org_purchases=2, site_visits=4, fit_quiz_completed=1, source='Meta paid social')

0.6692692202570364